# Regression Models and Gradient Descent

**Name : Dhruv Bompilwar**  
**PRN : 202401040200**  
**Roll no : 49**  
**Division : A**  
**Batch : A2**

### Aim

1. Develop regression models for a real world application and check their performance using suitable metrics.
2. Implement Gradient Descent for Linear Regression and study its performance.

### Problem taken: House Price Prediction

In this practical, house price is predicted using some basic details of a house like area, number of bedrooms, age and distance from city centre.

The same dataset is used for all the models so that the results can be compared.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

# creating a small house price dataset
n = 250

area = np.random.randint(500, 3001, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.randint(0, 31, n)
distance = np.round(np.random.uniform(1, 25, n), 2)

noise = np.random.normal(0, 8, n)

price = (
    0.055 * area
    + 9 * bedrooms
    - 0.9 * age
    - 1.5 * distance
    + 20
    + noise
)

df = pd.DataFrame({
    "Area": area,
    "Bedrooms": bedrooms,
    "Age": age,
    "Distance": distance,
    "Price": price
})

df.head()


### Looking at the data

First, we can check some basic information about the dataset and see the relation between area and price.

In [ ]:
print(df.shape)
display(df.describe())

plt.figure(figsize=(7, 5))
plt.scatter(df["Area"], df["Price"])
plt.xlabel("Area (sq ft)")
plt.ylabel("Price (Lakh)")
plt.title("Area vs Price")
plt.grid()
plt.show()


### Train and test data

80% data is used for training and 20% for testing. Scaling is done because the feature values have different ranges.

In [ ]:
X = df[["Area", "Bedrooms", "Age", "Distance"]]
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


### Evaluation metrics

The following metrics are used:

- **MAE:** average absolute error. Lower is better.
- **MSE:** average squared error. Lower is better.
- **RMSE:** square root of MSE. Lower is better.
- **R²:** shows how well the model explains the target values. Higher is better.

In [ ]:
def get_metrics(y_actual, y_pred):
    mae = mean_absolute_error(y_actual, y_pred)
    mse = mean_squared_error(y_actual, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_actual, y_pred)

    return mae, mse, rmse, r2


## 1. Multiple Linear Regression

Linear Regression is used as the basic model. It uses all four features to predict the house price.

In [ ]:
linear = LinearRegression()
linear.fit(X_train_scaled, y_train)

pred_linear = linear.predict(X_test_scaled)

mae, mse, rmse, r2 = get_metrics(y_test, pred_linear)

print("MAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)
print("R2   :", r2)


## 2. Ridge Regression

Ridge Regression is similar to Linear Regression but adds regularization. It can help when the model has large coefficients or correlated features.

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

pred_ridge = ridge.predict(X_test_scaled)

mae, mse, rmse, r2 = get_metrics(y_test, pred_ridge)

print("MAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)
print("R2   :", r2)


## 3. Polynomial Regression

Polynomial Regression is used to add some nonlinear terms to the model. Here, degree 2 is used.

In [ ]:
poly = PolynomialFeatures(degree=2)

X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

pred_poly = poly_model.predict(X_test_poly)

mae, mse, rmse, r2 = get_metrics(y_test, pred_poly)

print("MAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)
print("R2   :", r2)


## 4. Comparing the Regression Models

Now the results of the three models are put together. The model with lower RMSE and MAE and higher R² is better.

In [ ]:
results = []

for name, prediction in [
    ("Linear Regression", pred_linear),
    ("Ridge Regression", pred_ridge),
    ("Polynomial Regression", pred_poly)
]:
    mae, mse, rmse, r2 = get_metrics(y_test, prediction)

    results.append([
        name, mae, mse, rmse, r2
    ])

result_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "MSE", "RMSE", "R2"]
)

display(result_df)


### Actual and predicted prices

The points should be close to the diagonal line when the predictions are good.

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(y_test, pred_linear, label="Linear")
plt.scatter(y_test, pred_ridge, label="Ridge")
plt.scatter(y_test, pred_poly, label="Polynomial")

low = min(y_test.min(), pred_linear.min(), pred_ridge.min(), pred_poly.min())
high = max(y_test.max(), pred_linear.max(), pred_ridge.max(), pred_poly.max())

plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Price")
plt.legend()
plt.grid()
plt.show()


# Part 2: Gradient Descent

### Gradient Descent for Linear Regression

Gradient Descent starts with some initial values of weights and bias. It then changes them step by step to reduce the error.

The learning rate decides how big each step is. The cost used here is Mean Squared Error.

In [ ]:
class MyLinearRegression:

    def __init__(self, learning_rate=0.05, iterations=2000):
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = 0
        self.cost = []

    def fit(self, X, y):
        rows, columns = X.shape

        self.weights = np.zeros(columns)
        self.bias = 0
        self.cost = []

        for i in range(self.iterations):

            y_pred = np.dot(X, self.weights) + self.bias

            error = y_pred - y

            cost = np.mean(error ** 2)
            self.cost.append(cost)

            dw = (2 / rows) * np.dot(X.T, error)
            db = (2 / rows) * np.sum(error)

            self.weights = self.weights - self.learning_rate * dw
            self.bias = self.bias - self.learning_rate * db

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias


### Training the model using Gradient Descent

The same house price data is used here. The result can then be compared with the Linear Regression model above.

In [ ]:
gd = MyLinearRegression(
    learning_rate=0.05,
    iterations=2000
)

gd.fit(X_train_scaled, y_train)

pred_gd = gd.predict(X_test_scaled)

mae, mse, rmse, r2 = get_metrics(y_test, pred_gd)

print("MAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)
print("R2   :", r2)

print("\nWeights:", gd.weights)
print("Bias:", gd.bias)


### Cost vs iterations

If Gradient Descent is working properly, the cost should decrease as the number of iterations increases.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(gd.cost)
plt.xlabel("Iterations")
plt.ylabel("MSE")
plt.title("Gradient Descent Cost")
plt.grid()
plt.show()

print("Starting cost:", gd.cost[0])
print("Final cost:", gd.cost[-1])


### Effect of learning rate

Different learning rates are tested below. A very small value usually takes more iterations, while a suitable value reaches a low cost faster.

In [ ]:
rates = [0.001, 0.01, 0.05, 0.1]

rate_results = []

plt.figure(figsize=(8, 5))

for rate in rates:

    model = MyLinearRegression(
        learning_rate=rate,
        iterations=1000
    )

    model.fit(X_train_scaled, y_train)

    prediction = model.predict(X_test_scaled)

    mae, mse, rmse, r2 = get_metrics(y_test, prediction)

    rate_results.append([
        rate,
        model.cost[-1],
        mae,
        rmse,
        r2
    ])

    plt.plot(model.cost, label=str(rate))

plt.xlabel("Iterations")
plt.ylabel("MSE")
plt.title("Different Learning Rates")
plt.legend(title="Learning Rate")
plt.grid()
plt.show()

rate_df = pd.DataFrame(
    rate_results,
    columns=["Learning Rate", "Final Cost", "MAE", "RMSE", "R2"]
)

display(rate_df)


### Gradient Descent and Linear Regression comparison

Both methods are solving the same linear regression problem. If Gradient Descent has converged properly, its results should be close to the standard Linear Regression result.

In [ ]:
gd_result = get_metrics(y_test, pred_gd)
linear_result = get_metrics(y_test, pred_linear)

comparison = pd.DataFrame(
    [
        ["Linear Regression", *linear_result],
        ["Gradient Descent", *gd_result]
    ],
    columns=["Model", "MAE", "MSE", "RMSE", "R2"]
)

display(comparison)


## Conclusion

In this practical, house price prediction was taken as a real-world regression problem.

Linear Regression, Ridge Regression and Polynomial Regression were trained and compared using MAE, MSE, RMSE and R².

Gradient Descent was also implemented from scratch for Linear Regression. The cost decreased during training, showing that the model was learning. Different learning rates were tested to see their effect on convergence.

The final results show that Gradient Descent can give results close to standard Linear Regression when the learning rate and number of iterations are chosen properly.